In [ ]:
import os
import json
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import words
import tempfile
import os
import json
from transformers import AutoTokenizer
from collections import Counter
import math
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def generate_history(history):
    history_text = '\n'.join(
            [
                f"{message['content']}"
                for message in history
            ]
    )
    return history_text

In [ ]:
temp_dir = tempfile.mkdtemp()
nltk.download('punkt', download_dir=temp_dir)
nltk.download('punkt_tab',download_dir=temp_dir)
nltk.data.path.append(temp_dir)

In [ ]:
folder_path = 'path_to_sessions'

In [ ]:
nltk.download('words',download_dir=temp_dir)
vocab = set(words.words())
vocab_size = len(vocab)

In [ ]:
with open("../valid_indices.json", "r") as f:
    valid_indices = json.load(f)

In [10]:
fourgram = []
trigram = []
bigram = []
unigram = []

In [11]:
def remove_unwanted(document):

    # remove user mentions
    document = re.sub("@[A-Za-z0-9_]+"," ", document)
    # remove URLS
    document = re.sub(r'http\S+', ' ', document)
    # remove hashtags
    document = re.sub("#[A-Za-z0-9_]+","", document)
    # remove punctuation
    document = re.sub("[^0-9A-Za-z ]", "" , document)
    # remove double spaces
    document = document.replace('  '," ")
    
    return document.strip()

In [ ]:
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        x = int(filename.replace('session_', '').replace('.json', ''))
        if (x-1) in valid_indices:
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            conv_str = generate_history(json_data["history"])
            conv_mod = remove_unwanted(conv_str)
            conv_tokens = word_tokenize(conv_mod.lower())  # lowercase optional
            
            # Build n-grams
            conv_unigram = conv_tokens
            conv_bigram = list(zip(conv_tokens, conv_tokens[1:]))
            conv_trigram = list(zip(conv_tokens, conv_tokens[1:], conv_tokens[2:]))
            conv_fourgram = list(zip(conv_tokens, conv_tokens[1:], conv_tokens[2:], conv_tokens[3:]))
            
            # Append to global lists
            unigram.extend(conv_unigram)
            bigram.extend(conv_bigram)
            trigram.extend(conv_trigram)
            fourgram.extend(conv_fourgram)

In [ ]:
dist_3 = len(set(trigram))/len(trigram)
print(dist_3)

In [ ]:
dist_2 = len(set(bigram))/len(bigram)
print(dist_2)

In [ ]:
dist_1 = len(set(unigram))/len(unigram)
print(dist_1)

In [ ]:
ead = len(set(unigram))/(vocab_size*(1-((vocab_size-1)/vocab_size)**len(unigram)))
print(ead)

In [18]:
total_dialogues_client = 0
total_dialogues_counselor = 0
unique_words_dialogues_client = 0
unique_words_dialogues_counselor = 0
unigram_client = []
unigram_counselor = []

In [ ]:
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        x = int(filename.replace('session_', '').replace('.json', ''))
        if (x-1) in valid_indices:
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            client_list = []
            counselor_list = []
            for d in json_data["history"]:
                if d["role"] == "counselor":
                    counselor_list.append(d)
                    total_dialogues_counselor = total_dialogues_counselor + 1
                    diag_str = d["content"]
                    diag_mod = remove_unwanted(diag_str)
                    diag_tokens = word_tokenize(diag_mod.lower())
                    unique_words_dialogues_counselor = unique_words_dialogues_counselor + len(set(diag_tokens))
                elif d["role"] == "client":
                    client_list.append(d)
                    total_dialogues_client = total_dialogues_client + 1
                    diag_str = d["content"]
                    diag_mod = remove_unwanted(diag_str)
                    diag_tokens = word_tokenize(diag_mod.lower())
                    unique_words_dialogues_client = unique_words_dialogues_client + len(set(diag_tokens))
            conv_str_client = generate_history(client_list)
            conv_str_counselor = generate_history(counselor_list)
            conv_mod_client = remove_unwanted(conv_str_client)
            conv_mod_counselor = remove_unwanted(conv_str_counselor)
            conv_tokens_client = word_tokenize(conv_mod_client.lower())  # lowercase optional
            conv_tokens_counselor = word_tokenize(conv_mod_counselor.lower())  # lowercase optional
            
            # Build n-grams
            conv_unigram_client = conv_tokens_client
            conv_unigram_counselor = conv_tokens_counselor
            
            # Append to global lists
            unigram_client.extend(conv_unigram_client)
            unigram_counselor.extend(conv_unigram_counselor)

In [ ]:
print(unique_words_dialogues_client/total_dialogues_client)
print(len(set(unigram_client))*100/len(unigram_client))

In [ ]:
ldd_client = (unique_words_dialogues_client/total_dialogues_client)*(len(set(unigram_client))/len(unigram_client))*100
print(ldd_client)

In [ ]:
print(unique_words_dialogues_counselor/total_dialogues_counselor)
print(len(set(unigram_counselor))*100/len(unigram_counselor))

In [ ]:
ldd_counselor = (unique_words_dialogues_counselor/total_dialogues_counselor)*(len(set(unigram_counselor))/len(unigram_counselor))*100
print(ldd_counselor)

In [24]:
def entropy(lst, base=math.e):
    counts = Counter(lst)
    total = len(lst)

    entropy_val = 0.0
    for count in counts.values():
        p = count / total
        entropy_val -= p * math.log(p, base)

    return entropy_val, counts

In [ ]:
ent1,_ = entropy(unigram)
print(ent1)

In [ ]:
ent2,_ = entropy(bigram)
print(ent2)

In [ ]:
ent3,_ = entropy(trigram)
print(ent3)

In [ ]:
ent4,_ = entropy(fourgram)
print(ent4)

In [ ]:
with open('path_to_situation_extractions', 'r') as file:
    situation_dict = json.load(file)

In [30]:
situations = []

In [31]:
for key in situation_dict.keys():
    x = int(key.split('_')[1])
    if (x-1) in valid_indices:
        situations.append(situation_dict[key])

In [32]:
from sentence_transformers import SentenceTransformer

In [ ]:
attn_implementation = "eager" 
model = SentenceTransformer(
    "nvidia/llama-embed-nemotron-8b",
    trust_remote_code=True,
    model_kwargs={"attn_implementation": attn_implementation, "torch_dtype": "bfloat16"},
    tokenizer_kwargs={"padding_side": "left"},
)

In [ ]:
situation_embeddings = model.encode_document(situations)
N = situation_embeddings.shape[0]
sim_matrix = cosine_similarity(situation_embeddings)
upper_tri_indices = np.triu_indices(N, k=1)
avg_pairwise_similarity = sim_matrix[upper_tri_indices].mean()

In [ ]:
print(avg_pairwise_similarity)